# Targeted Speech Adversarial Inspection

Inspect targeted Speech attacks (`target_audioset_label=0`) for:

- `beats_iter3_plus_as2m`
- `audiomae_as2m_ft_as20k`
- `panns_cnn14`

The notebook loads generated outputs, plays original/adversarial clips, plots model-input
cochleagram-like features, and checks epsilon-budget compliance.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Audio, Markdown, display

from src.models.audio import get_audio_model

MODEL_NAMES = [
    "beats_iter3_plus_as2m",
    "audiomae_as2m_ft_as20k",
    "panns_cnn14",
]
EPSILONS = [2.4, 2.8]
TARGET_AUDIOSET_LABEL = 0
TARGET_SUFFIX = "targeted_label0000"

DEVICE = "cpu"

_MODEL_CACHE: dict[str, Any] = {}


def _resolve_exp_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "experiments" / "audio",
        cwd.parent / "experiments" / "audio",
        Path("/orcd/data/jhm/001/om2/rphess/projects/github.com/model_metamers_pytorch/experiments/audio"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise RuntimeError(
        "Could not locate experiments/audio. Tried: "
        + ", ".join(str(path) for path in candidates)
    )


EXP_ROOT = _resolve_exp_root()
display(Markdown(f"Using experiment root: `{EXP_ROOT}`"))


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records


def to_feature_2d(wrapper: Any, model_name: str, waveform: torch.Tensor, sr: int) -> np.ndarray:
    waveform = waveform.detach().cpu().float()
    if model_name == "panns_cnn14":
        feature = wrapper.input_log_mel_spectrogram(waveform, sr=sr)
    else:
        feature = wrapper.preprocess(waveform, sr=sr)
    arr = feature.detach().cpu().numpy()
    while arr.ndim > 2:
        arr = arr[0]
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D feature, got shape {arr.shape}")
    if arr.shape[0] > arr.shape[1]:
        arr = arr.T
    return arr


def get_model(model_name: str):
    if model_name not in _MODEL_CACHE:
        _MODEL_CACHE[model_name] = get_audio_model(model_name, device=DEVICE, freeze=True)
    return _MODEL_CACHE[model_name]


def resolve_metadata_paths() -> dict[str, dict[float, Path]]:
    result: dict[str, dict[float, Path]] = {}
    for model_name in MODEL_NAMES:
        per_eps: dict[float, Path] = {}
        for eps in EPSILONS:
            eps_token = format(float(eps), "g").replace(".", "p")
            attack_id = f"adversarial_l2_eps_{eps_token}_{TARGET_SUFFIX}"
            path = EXP_ROOT / model_name / "adversarial" / attack_id / "logits" / "metadata.jsonl"
            if path.exists():
                per_eps[float(eps)] = path
        result[model_name] = per_eps
    return result


def collect_records() -> dict[str, dict[float, dict[int, dict[str, Any]]]]:
    records: dict[str, dict[float, dict[int, dict[str, Any]]]] = {}
    paths = resolve_metadata_paths()
    for model_name, per_eps in paths.items():
        records[model_name] = {}
        for eps, meta_path in per_eps.items():
            indexed: dict[int, dict[str, Any]] = {}
            for record in load_jsonl(meta_path):
                if int(record.get("target_audioset_label", -1)) != TARGET_AUDIOSET_LABEL:
                    continue
                sample_idx = int(record["sample_idx"])
                indexed[sample_idx] = record
            records[model_name][eps] = indexed
    return records


Using experiment root: `/orcd/data/jhm/001/om2/rphess/projects/github.com/model_metamers_pytorch/experiments/audio`

In [2]:
records = collect_records()
paths = resolve_metadata_paths()

available_pairs = [
    (model_name, float(eps))
    for model_name in MODEL_NAMES
    for eps in EPSILONS
    if eps in records.get(model_name, {})
]

if not available_pairs:
    expected = []
    for model_name in MODEL_NAMES:
        for eps in EPSILONS:
            eps_token = format(float(eps), "g").replace(".", "p")
            attack_id = f"adversarial_l2_eps_{eps_token}_{TARGET_SUFFIX}"
            expected.append(
                EXP_ROOT / model_name / "adversarial" / attack_id / "logits" / "metadata.jsonl"
            )
    raise RuntimeError(
        "No targeted metadata files were found. Expected paths include: "
        + "; ".join(str(path) for path in expected)
    )

shared_indices: set[int] | None = None
for model_name, eps in available_pairs:
    idxs = set(records[model_name][eps].keys())
    shared_indices = idxs if shared_indices is None else shared_indices.intersection(idxs)

if not shared_indices:
    pair_counts = {
        f"{model_name}@{eps}": len(records[model_name][eps])
        for model_name, eps in available_pairs
    }
    raise RuntimeError(
        "No shared targeted examples found across loaded model/epsilon pairs. "
        f"Pair counts: {pair_counts}"
    )

shared_indices = sorted(shared_indices)
display(Markdown(f"**Loaded pairs:** `{available_pairs}`"))
display(Markdown(f"**Shared indices across loaded runs:** `{shared_indices}`"))


def recompute_budget(original: torch.Tensor, adversarial: torch.Tensor, norm: str, epsilon: float) -> dict[str, Any]:
    delta = (adversarial - original).detach().view(original.shape[0], -1)
    if norm == "l2":
        active = torch.norm(delta, p=2, dim=1)
    elif norm == "linf":
        active = torch.norm(delta, p=float("inf"), dim=1)
    else:
        raise ValueError(f"Unsupported norm {norm}")
    max_norm = float(active.max().item())
    return {
        "max_active_norm": max_norm,
        "within_budget": bool(max_norm <= (float(epsilon) + 1e-6)),
    }


def render_example(model_name: str, epsilon: float, sample_idx: int) -> None:
    record = records[model_name][epsilon][sample_idx]
    saved_paths = record["saved_paths"]
    original = torch.load(saved_paths["original_pt"], map_location="cpu")
    adversarial = torch.load(saved_paths["adversarial_pt"], map_location="cpu")
    delta = adversarial - original
    sr = int(record.get("sample_rate", 16000))

    norm = str(record.get("attack_config", {}).get("norm", "l2"))
    epsilon_value = float(record.get("attack_config", {}).get("epsilon", epsilon))
    recomputed = recompute_budget(original, adversarial, norm=norm, epsilon=epsilon_value)
    stored_budget = record.get("saved_output_budget_verification", {})

    display(Markdown(f"### {model_name} | eps={epsilon} | sample={sample_idx}"))
    display(Markdown(
        f"- Stored budget ok: `{stored_budget.get('within_budget', 'n/a')}`  \n"
        f"- Recomputed budget ok: `{recomputed['within_budget']}`  \n"
        f"- Recomputed max norm: `{recomputed['max_active_norm']:.6f}`"
    ))

    orig_np = original.squeeze().detach().cpu().numpy()
    adv_np = adversarial.squeeze().detach().cpu().numpy()
    delta_np = delta.squeeze().detach().cpu().numpy()

    display(Markdown("**Original audio**"))
    display(Audio(orig_np, rate=sr))
    display(Markdown("**Targeted adversarial audio**"))
    display(Audio(adv_np, rate=sr))
    display(Markdown("**Perturbation (amplified for listening)**"))
    peak = np.max(np.abs(delta_np))
    scaled_delta = delta_np if peak < 1e-8 else 0.95 * delta_np / peak
    display(Audio(scaled_delta, rate=sr))

    wrapper = get_model(model_name)
    feat_orig = to_feature_2d(wrapper, model_name, original, sr)
    feat_adv = to_feature_2d(wrapper, model_name, adversarial, sr)
    feat_diff = feat_adv - feat_orig

    fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
    axes[0].imshow(feat_orig, aspect="auto", origin="lower", cmap="magma")
    axes[0].set_title("Original feature")
    axes[1].imshow(feat_adv, aspect="auto", origin="lower", cmap="magma")
    axes[1].set_title("Targeted adversarial feature")
    vmax = np.percentile(np.abs(feat_diff), 99)
    vmax = float(max(vmax, 1e-8))
    axes[2].imshow(
        feat_diff,
        aspect="auto",
        origin="lower",
        cmap="coolwarm",
        vmin=-vmax,
        vmax=vmax,
    )
    axes[2].set_title("Feature difference (adv - orig)")
    for ax in axes:
        ax.set_xlabel("Time bins")
        ax.set_ylabel("Frequency bins")
    plt.show()


**Loaded pairs:** `[('beats_iter3_plus_as2m', 2.4), ('beats_iter3_plus_as2m', 2.8), ('audiomae_as2m_ft_as20k', 2.4), ('audiomae_as2m_ft_as20k', 2.8), ('panns_cnn14', 2.4), ('panns_cnn14', 2.8)]`

**Shared indices across loaded runs:** `[0, 5, 10, 11, 12, 14, 19, 20, 22, 23]`

In [ ]:
available_models = sorted([model for model in MODEL_NAMES if records.get(model)])
if not available_models:
    raise RuntimeError("No models with loaded targeted records are available for widgets.")

model_widget = widgets.Dropdown(
    options=available_models,
    value=available_models[0],
    description="model",
)
epsilon_widget = widgets.Dropdown(options=[], description="epsilon")
sample_widget = widgets.Dropdown(options=[], description="sample")


def _sync_eps_options(*_args):
    selected_model = model_widget.value
    available_eps = sorted(records.get(selected_model, {}).keys())
    epsilon_widget.options = available_eps
    if available_eps and epsilon_widget.value not in available_eps:
        epsilon_widget.value = available_eps[0]


def _sync_sample_options(*_args):
    selected_model = model_widget.value
    selected_eps = float(epsilon_widget.value)
    available = sorted(records.get(selected_model, {}).get(selected_eps, {}).keys())
    sample_widget.options = available
    if available and sample_widget.value not in available:
        sample_widget.value = available[0]


model_widget.observe(_sync_eps_options, names="value")
model_widget.observe(_sync_sample_options, names="value")
epsilon_widget.observe(_sync_sample_options, names="value")
_sync_eps_options()
_sync_sample_options()

ui = widgets.HBox([model_widget, epsilon_widget, sample_widget])
out = widgets.interactive_output(
    lambda model, epsilon, sample: render_example(model, float(epsilon), int(sample)),
    {"model": model_widget, "epsilon": epsilon_widget, "sample": sample_widget},
)
display(ui, out)

Output()